In [10]:
pwd

'/home/gmvaz/2026_GEMs/stm_model_test'

In [26]:
import os
import pandas as pd

import cobra
from cobra.core import Configuration
from cobra.io import load_json_model

#iMATpy
from imatpy.imat import imat
from imatpy.model_creation import generate_model
from imatpy.parse_gpr import gene_to_rxn_weights
from imatpy.model_utils import model_eq

In [3]:
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [4]:
# Load the input model of Salmonella Tm. - stm_v1_0
stm_v1 = load_json_model("STM_v1_0.json")
stm_v1.solver = "gurobi"

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [11]:
# read in the csv files of the threshold rna-seq data
threshold_csv = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

In [12]:
base_model = stm_v1

In [13]:
# python set
threshold_genes = set(threshold_csv["genes"].astype(str).str.strip())

# make a list of genes and save it as a set
model_genes = set([g.id for g in base_model.genes])

# find genes that exist in the model and RNA-seq
overlap = threshold_genes.intersection(model_genes)

# find genes that appear in the RNA-Seq data but are not in the model
only_stm = threshold_genes.difference(model_genes)

# find genes that exist in the model but do not appear in RNA-Seq data
only_model = model_genes.difference(threshold_genes)

# subset the rna-seq expression data to only have the genes present in the model
threshold_model = threshold_csv[threshold_csv["genes"].isin(model_genes)]

# to this subset - add the 20 genes present in the model but missing in the rna-seq data and assign a value of 0
# convert from dataframe to panda series, make sure all the genes from the model exist, and fill missing values with 0
expression_subset = threshold_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

# convert gene weights to reaction weights
rxn_weights = gene_to_rxn_weights(base_model, expression_subset)


In [ ]:
model = generate_model(model = base_model, rxn_weights = rxn_weights, method = 'imat_restrictions')
#save_json_model(
#    model,
#    args.output
#    )

In [16]:
pwd

'/home/gmvaz/2026_GEMs/stm_model_test'

In [17]:
model

Name,STM_v1_0
Memory address,11e330081690
Number of metabolites,1802
Number of reactions,2545
Number of genes,1271
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Compartments,"cytosol, periplasm, extracellular space"


In [18]:
expression_subset

genes
STM3926    0
STM2818    1
STM3883    0
STM3692    0
STM4227    0
          ..
STM1570    1
STM2401    0
STM0054   -1
STM2415    1
STM1000    1
Name: threshold_category, Length: 1271, dtype: int64

In [19]:
rxn_weights

3HAD180      1.0
3HAD181      1.0
3HAD40       1.0
3HAD60       1.0
3HAD80       1.0
            ... 
12dgr2_ST    0.0
OAL_ST       0.0
OA4L_ST      0.0
OA5L_ST      0.0
OA4VL_ST     0.0
Length: 2545, dtype: float64

In [28]:
list(rxn_weights.items())

[('3HAD180', 1.0),
 ('3HAD181', 1.0),
 ('3HAD40', 1.0),
 ('3HAD60', 1.0),
 ('3HAD80', 1.0),
 ('3KGK', 0.0),
 ('3NTD2pp', 1.0),
 ('3NTD4pp', 1.0),
 ('3NTD7pp', 1.0),
 ('3NTD9pp', 1.0),
 ('3OAR100', 1.0),
 ('3OAR120', 1.0),
 ('3OAR121', 1.0),
 ('3OAR140', 1.0),
 ('3OAR141', 1.0),
 ('3OAR160', 1.0),
 ('3OAR161', 1.0),
 ('23CAMPtex', 1.0),
 ('23CCMPtex', 1.0),
 ('23CGMPtex', 1.0),
 ('12DGR120tipp', 0.0),
 ('12DGR140tipp', 0.0),
 ('12DGR141tipp', 0.0),
 ('12DGR160tipp', 0.0),
 ('23CUMPtex', 1.0),
 ('3OAR180', 1.0),
 ('23DAPPAt2pp', 0.0),
 ('23DAPPAtex', 1.0),
 ('12DGR161tipp', 0.0),
 ('12DGR180tipp', 0.0),
 ('12DGR181tipp', 0.0),
 ('12PPDRtex', 1.0),
 ('12PPDRtpp', 1.0),
 ('12PPDStex', 1.0),
 ('12PPDStpp', 1.0),
 ('23PDE2pp', 1.0),
 ('23PDE4pp', 1.0),
 ('23PDE7pp', 1.0),
 ('23PDE9pp', 1.0),
 ('26DAHtex', 1.0),
 ('2AGPA120tipp', 0.0),
 ('2AGPA140tipp', 0.0),
 ('2AGPA141tipp', 0.0),
 ('2AGPA160tipp', 0.0),
 ('2AGPA161tipp', 0.0),
 ('2AGPA180tipp', 0.0),
 ('2AGPA181tipp', 0.0),
 ('14GLUCANabcp

In [29]:
rxn_df = rxn_weights.to_frame()

In [31]:
rxn_df.columns

RangeIndex(start=0, stop=1, step=1)

In [34]:
rxn_df = rxn_df.rename(columns = {0:"weight"})

In [37]:
print(len(rxn_df[rxn_df.weight == 0]))
print(len(rxn_df[rxn_df.weight == 1]))
print(len(rxn_df[rxn_df.weight == -1]))

1472
965
108


In [20]:
len(model_genes)

1271

In [21]:
len(overlap)

1251

In [22]:
len(only_stm)

3420

In [23]:
len(only_model)

20

In [24]:
expression_subset

genes
STM3926    0
STM2818    1
STM3883    0
STM3692    0
STM4227    0
          ..
STM1570    1
STM2401    0
STM0054   -1
STM2415    1
STM1000    1
Name: threshold_category, Length: 1271, dtype: int64

In [25]:
model == base_model

False

In [27]:
model_eq(model, base_model, verbose = True)

Verbose model comparison
Models have different constraints


False